## Preliminares

In [14]:
# Importar modulos y funciones necesarias
import pandas as pd
import numpy as np
import os
from datetime import datetime
from sklearn.model_selection import cross_val_score, GridSearchCV, TimeSeriesSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import plotly.express as px
from src.evaluators import *

In [15]:
# Abrir archivo clean_data
data_folder = "data"
df = pd.read_parquet(f"{data_folder}/clean_data.parquet")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17986 entries, 0 to 17985
Data columns (total 37 columns):
 #   Column                          Non-Null Count  Dtype   
---  ------                          --------------  -----   
 0   YearsSinceAdded                 17986 non-null  float64 
 1   Monthly_Return                  17986 non-null  float64 
 2   Monthly_Variance                17986 non-null  float64 
 3   Market_Covariance               17986 non-null  float64 
 4   operatingMargins                17986 non-null  float64 
 5   profitMargins                   17986 non-null  float64 
 6   ReturnOnAssets                  17986 non-null  float64 
 7   Revenue_YoY                     17986 non-null  float64 
 8   Revenue_QoQ                     17986 non-null  float64 
 9   EBITDA_YoY                      17986 non-null  float64 
 10  EBITDA_QoQ                      17986 non-null  float64 
 11  FCF_YoY                         17986 non-null  float64 
 12  FCF_QoQ           

## Feature Engineering

Sección para crear variables en la fase de modelado. 
La mayor parte de las variables fue creada en la fase de extracción.

In [16]:
# Calcular las aceleraciones netas (Momento - Tendencia) para variables de crecimiento.
# Una vez calculadas, se eliminan las variables trimestrales (reemplazar=True)
variables_de_crecimiento = ['CPI_QoQ', 'M2_QoQ', 'Revenue_QoQ', 'EBITDA_QoQ', 'FCF_QoQ', 'Capex_QoQ']
df = calcular_acceleration_features(df, variables_de_crecimiento, reemplazar= True)

In [17]:
# Sectores poco significativos: 
# Solo InformationTechnology resulta relevante, y Energy marginalmente.
# se agrupan en la categoria "Other" el resto de los sectores
sectores_importantes = ['InformationTechnology', 'Energy']

df['Sector'] = np.where(df['Sector'].isin(sectores_importantes), df['Sector'], 'Other')

# Se vuelve a convertir en category
df['Sector'] = df['Sector'].astype('category')

In [18]:
# Se eliminan columnas con significancia menor a 0.01
df = df.drop(columns=[
    'CPI_YoY',
    'CPI_Acceleration',
    'Monthly_Return',
    'UnemploymentRate',
    '10Y2YSpread',
    'HighYieldSpread',
    'FedFundsRate',
    'M2_Acceleration',
    'M2_YoY',
    'Market_Covariance'
])

# Modelo de ensamblado de árboles RandomForest

In [19]:
# Ratios de valuación
ratios = ['PriceToBook_Transformed', 'PE_Trailing_Transformed', 'EnterpriseToEbitda_Transformed']

# Se asegura el ordenamiento por fecha
df.sort_values(by='Date', inplace=True)

# Se define la variable objetivo y las variables predictoras
label = 'EnterpriseToEbitda_Transformed'
X = df.drop(columns=ratios + ['Ticker', 'Date']) # Se quitan los ratios de valuación restantes, Ticker y Fecha de las variables predictoras
y = df[label]

# Columnas numéricas: 
numeric_cols = X.select_dtypes(include=np.number).columns.tolist()

# Variables categóricas:
categorical_cols = ['Sector']

# preprocesador: escala numéricas y codifica categóricas
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
])

pipe = Pipeline([
    ('pre', preprocessor),
    ('model', RandomForestRegressor(
        random_state=42,
        n_estimators=300,
        max_depth=10,
        min_samples_leaf=10,
        max_features='sqrt',
        max_samples= 0.8,
        min_samples_split= 10         
        ))
])

print("Entrenando el modelo con datos completos...")
pipe.fit(X, y)
r2_completo = pipe.score(X, y)
print(f"Entrenamiento finalizado. R2 en datos completos: {r2_completo:.4f}")

Entrenando el modelo con datos completos...
Entrenamiento finalizado. R2 en datos completos: 0.8117


In [20]:
# Test de validación cruzada
# Configurar el validador de series temporales
tscv = TimeSeriesSplit(n_splits=5) # n_splits=5 creará 5 particiones temporales secuenciales

# 3. Test de validación cruzada temporal
cross_val_scores = cross_val_score(
    estimator=pipe, 
    X=X, 
    y=y, 
    cv=tscv,         
    scoring='r2',
    n_jobs=-1        
)

print(f"R² promedio Time Series CV: {cross_val_scores.mean():.4f} ± {cross_val_scores.std():.4f}")

R² promedio Time Series CV: 0.5916 ± 0.0865


In [21]:
# Importancia de factores en el modelo
rf_model = pipe.named_steps['model']
importances = rf_model.feature_importances_

# Obtener los nombres de las características después del preprocesamiento
preprocessor = pipe.named_steps['pre']
feature_names = preprocessor.get_feature_names_out()

feature_importance_df = pd.DataFrame({'feature': feature_names, 'importance': importances})
feature_importance_df = feature_importance_df.sort_values(by='importance', ascending=False)
feature_importance_df.head(20)

,feature,importance
10,num__FCF_to_EBITDA,0.136009
9,num__NetDebt_to_EBITDA,0.099057
17,num__Revenue_Acceleration,0.088191
5,num__Revenue_YoY,0.086839
4,num__ReturnOnAssets,0.064475
13,num__MarketCap_log,0.050793
2,num__operatingMargins,0.046000
3,num__profitMargins,0.044927
0,num__YearsSinceAdded,0.038848
22,cat__Sector_InformationTechnology,0.038645


## Aplicacion del modelo

Se generan 2 clusters segun las predicciones:
* Positive_Bias: si los residuos son mayores o iguales a cero.
* Negative_Bias: si los residuos son menores a cero.

In [22]:
# Se dividen los datos para predecir el valor de la última fecha disponible para cada ticker en el conjunto de test
X_train = X.iloc[:-len(df['Ticker'].unique())]  # Todos menos la última fecha de cada ticker
y_train = y.iloc[:-len(df['Ticker'].unique())]
X_test = X.iloc[-len(df['Ticker'].unique()):]   # Solo la última fecha de cada ticker
y_test = y.iloc[-len(df['Ticker'].unique()):]

# Recuperar el ticker usando el indice de y_test
tickers_test = df.loc[y_test.index, 'Ticker']

# Predicciones en el conjunto de test
y_pred = pipe.predict(X_test)

# Consolidar resultados individuales en un DataFrame
resultados_df = pd.DataFrame({
    'Ticker': tickers_test,
    'Predicho': y_pred,
    'Real': y_test
})

# Calcular el residuo para cada predicción
resultados_df['Residuo'] = resultados_df['Predicho'] - resultados_df['Real']

# Agrupar por ticker
resultados_agrupados = resultados_df.groupby('Ticker')[['Predicho', 'Real', 'Residuo']].mean()

# Generar el Cluster sobre el promedio de los residuos
resultados_agrupados['Cluster'] = ['Positive_Bias' if r >= 0 else 'Negative_Bias' 
                                   for r in resultados_agrupados['Residuo']]

# Visualizar
fig = px.scatter(
    resultados_agrupados, 
    x='Real', 
    y='Predicho', 
    color='Cluster',
    hover_name=resultados_agrupados.index, 
    labels={'Real':'Valor Real Promedio', 'Predicho':'Predicción Promedio', 'Cluster':'Sesgo del Modelo'},
    title='Predicciones vs Reales (Agrupado por Ticker)'
)

# Línea de identidad perfecta
min_val = resultados_agrupados['Real'].min()
max_val = resultados_agrupados['Real'].max()
fig.add_shape(type='line', x0=min_val, y0=min_val, x1=max_val, y1=max_val,
              line=dict(color='black', dash='dash', width=2))
fig.show()

# Estadísticas por cluster a nivel Ticker
over_mask = resultados_agrupados['Cluster'] == 'Positive_Bias'
under_mask = resultados_agrupados['Cluster'] == 'Negative_Bias'

print("\nEstadísticas por cluster (a nivel de Ticker):")
print(f"Overprediction: {over_mask.sum()} tickers, residuo medio global: {resultados_agrupados.loc[over_mask, 'Residuo'].mean():.4f}")
print(f"Underprediction: {under_mask.sum()} tickers, residuo medio global: {resultados_agrupados.loc[under_mask, 'Residuo'].mean():.4f}")


Estadísticas por cluster (a nivel de Ticker):
Overprediction: 267 tickers, residuo medio global: 0.0500
Underprediction: 186 tickers, residuo medio global: -0.0673


In [23]:
# Ordenar resultados por residuos
resultados_agrupados = resultados_agrupados.sort_values(by='Residuo', ascending=False)
# Positive Biases: Sobrepredicciones seleccionando solo valores positivos para los ratios Reales y Predichos
positive_biases = resultados_agrupados[(resultados_agrupados['Real'] > 0) & (resultados_agrupados['Predicho'] > 0) & (resultados_agrupados['Cluster'] == 'Positive_Bias')]

positive_biases.head(20)

,Predicho,Real,Residuo,Cluster
Ticker,,,,
WDAY,0.196367,0.034355,0.162012,Positive_Bias
DELL,0.152407,0.023881,0.128526,Positive_Bias
NOW,0.255504,0.137955,0.117549,Positive_Bias
NVDA,0.226030,0.137575,0.088455,Positive_Bias
APP,0.245050,0.184675,0.060375,Positive_Bias
APH,0.119108,0.063543,0.055565,Positive_Bias
MA,0.046414,0.000653,0.045760,Positive_Bias
GOOGL,0.045580,0.024554,0.021027,Positive_Bias
GOOG,0.042266,0.023433,0.018833,Positive_Bias


In [24]:
# Establer Ticker como columna para exportar resultados
positive_biases.reset_index(inplace=True)
positive_biases.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16 entries, 0 to 15
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Ticker    16 non-null     object 
 1   Predicho  16 non-null     float64
 2   Real      16 non-null     float64
 3   Residuo   16 non-null     float64
 4   Cluster   16 non-null     object 
dtypes: float64(3), object(2)
memory usage: 772.0+ bytes


In [25]:
# Se exportan los sesgos positivos a un archivo CSV para su análisis posterior 
# Guardar dataframe con cluster con fecha en el nombre del archivo

dia = datetime.now().day
mes = datetime.now().month
year = datetime.now().year

os.makedirs(f"{data_folder}/results", exist_ok=True) # Crear carpeta si no existe

positive_biases.to_csv(f"{data_folder}/results/positive_biases_{year}_{mes}_{dia}.csv", index=False)

## Anexo: optimización de hiperparámetros

In [26]:
ejecutar_celda = False

if ejecutar_celda:
    nombre_modelo = "Random Forest"
    print(f"Configurando GridSearchCV para {nombre_modelo}")

    # Pipeline usando el preprocesador específico para Random Forest
    modelo_base = Pipeline(steps=[
        ('preprocesador', preprocessor),
        ('rf', RandomForestRegressor(random_state=42))
    ])

    param_grid = {
        'rf__n_estimators': [300],
        'rf__max_depth': [10],
        'rf__min_samples_leaf': [10],
        'rf__min_samples_split': [10],
        'rf__max_samples': [0.8],
        'rf__max_features': ['sqrt']
    }

    # Configurar el GridSearchCV
    grid_search = GridSearchCV(
        estimator=modelo_base,
        param_grid=param_grid,
        scoring='r2',
        cv=tscv,
        n_jobs=-1,
        verbose=2
    )

    # Entrenar con datos completos
    print(f"Iniciando búsqueda de hiperparámetros. Esto tomará unos minutos.")
    grid_search.fit(X, y)

    # Resultados
    print("\n--- Búsqueda Finalizada ---")
    print("Mejores hiperparámetros encontrados:")
    print(grid_search.best_params_)